In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from tqdm import tqdm

In [3]:
BASE_PATH = "/content/drive/MyDrive/CIC_IoT_2023"
OUTPUT_PATH = f"{BASE_PATH}/Full_Dataset"

os.makedirs(OUTPUT_PATH, exist_ok=True)

LABEL_COL = "label"        # change ONLY if your dataset uses a different name
CHUNK_SIZE = 100_000       # safe for Colab
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

In [4]:
csv_files = [
    os.path.join(BASE_PATH, f)
    for f in os.listdir(BASE_PATH)
    if f.endswith(".csv")
]

print(f"✅ Found {len(csv_files)} CSV files")

✅ Found 169 CSV files


## 🟩 CELL 4 — PASS 1: Count labels across ALL files (STREAMING)

### 💡 This answers: “How many REAL samples do we have per class?”

In [5]:
global_label_counts = Counter()

for file in tqdm(csv_files, desc="Scanning label counts"):
    for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):
        labels = chunk[LABEL_COL].value_counts()
        global_label_counts.update(labels.to_dict())

label_counts_df = (
    pd.DataFrame.from_dict(global_label_counts, orient="index", columns=["count"])
      .sort_values("count")
)

label_counts_df

Scanning label counts: 100%|██████████| 169/169 [08:21<00:00,  2.97s/it]


,count
Uploading_Attack,1252
Recon-PingSweep,2262
Backdoor_Malware,3218
XSS,3846
SqlInjection,5245
CommandInjection,5409
BrowserHijacking,5859
DictionaryBruteForce,13064
DDoS-SlowLoris,23426
DDoS-HTTP_Flood,28790


In [6]:
# Sampling caps
KEEP_ALL_THRESHOLD = 200_000
MID_CAP = 40_000
HIGH_CAP = 30_000

MAX_TOTAL_TRAIN_ROWS = 900_000

# Test split ratio
TEST_RATIO = 0.2
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

In [7]:
class_caps = {}

for label, count in global_label_counts.items():
    if count < KEEP_ALL_THRESHOLD:
        class_caps[label] = count  # keep all
    elif count < 1_000_000:
        class_caps[label] = MID_CAP
    else:
        class_caps[label] = HIGH_CAP

class_caps

{'DDoS-ICMP_Flood': 30000,
 'DDoS-UDP_Flood': 30000,
 'DDoS-TCP_Flood': 30000,
 'DDoS-PSHACK_Flood': 30000,
 'DDoS-SYN_Flood': 30000,
 'DDoS-RSTFINFlood': 30000,
 'DDoS-SynonymousIP_Flood': 30000,
 'DoS-UDP_Flood': 30000,
 'DoS-TCP_Flood': 30000,
 'DoS-SYN_Flood': 30000,
 'BenignTraffic': 30000,
 'Mirai-greeth_flood': 40000,
 'Mirai-udpplain': 40000,
 'Mirai-greip_flood': 40000,
 'DDoS-ICMP_Fragmentation': 40000,
 'MITM-ArpSpoofing': 40000,
 'DDoS-UDP_Fragmentation': 40000,
 'DDoS-ACK_Fragmentation': 40000,
 'DNS_Spoofing': 178911,
 'Recon-HostDiscovery': 134378,
 'Recon-OSScan': 98259,
 'Recon-PortScan': 82284,
 'DoS-HTTP_Flood': 71864,
 'VulnerabilityScan': 37382,
 'DDoS-HTTP_Flood': 28790,
 'DDoS-SlowLoris': 23426,
 'DictionaryBruteForce': 13064,
 'SqlInjection': 5245,
 'BrowserHijacking': 5859,
 'CommandInjection': 5409,
 'Backdoor_Malware': 3218,
 'XSS': 3846,
 'Uploading_Attack': 1252,
 'Recon-PingSweep': 2262}

In [8]:
train_quota = {}
test_quota = {}

for label, cap in class_caps.items():
    test_quota[label] = int(cap * TEST_RATIO)
    train_quota[label] = cap - test_quota[label]

train_quota, test_quota

({'DDoS-ICMP_Flood': 24000,
  'DDoS-UDP_Flood': 24000,
  'DDoS-TCP_Flood': 24000,
  'DDoS-PSHACK_Flood': 24000,
  'DDoS-SYN_Flood': 24000,
  'DDoS-RSTFINFlood': 24000,
  'DDoS-SynonymousIP_Flood': 24000,
  'DoS-UDP_Flood': 24000,
  'DoS-TCP_Flood': 24000,
  'DoS-SYN_Flood': 24000,
  'BenignTraffic': 24000,
  'Mirai-greeth_flood': 32000,
  'Mirai-udpplain': 32000,
  'Mirai-greip_flood': 32000,
  'DDoS-ICMP_Fragmentation': 32000,
  'MITM-ArpSpoofing': 32000,
  'DDoS-UDP_Fragmentation': 32000,
  'DDoS-ACK_Fragmentation': 32000,
  'DNS_Spoofing': 143129,
  'Recon-HostDiscovery': 107503,
  'Recon-OSScan': 78608,
  'Recon-PortScan': 65828,
  'DoS-HTTP_Flood': 57492,
  'VulnerabilityScan': 29906,
  'DDoS-HTTP_Flood': 23032,
  'DDoS-SlowLoris': 18741,
  'DictionaryBruteForce': 10452,
  'SqlInjection': 4196,
  'BrowserHijacking': 4688,
  'CommandInjection': 4328,
  'Backdoor_Malware': 2575,
  'XSS': 3077,
  'Uploading_Attack': 1002,
  'Recon-PingSweep': 1810},
 {'DDoS-ICMP_Flood': 6000,
  'DDoS

In [9]:
expected_train_size = sum(train_quota.values())
print("Expected training size:", expected_train_size)

Expected training size: 1044367


In [10]:
OUTPUT_PATH = "/content/drive/MyDrive/CIC_IoT_2023/Full_Dataset"

train_path = f"{OUTPUT_PATH}/train_working.csv"
test_path = f"{OUTPUT_PATH}/test_frozen.csv"

# get columns safely
sample_df = pd.read_csv(csv_files[0], nrows=1)
columns = sample_df.columns.tolist()

pd.DataFrame(columns=columns).to_csv(train_path, index=False)
pd.DataFrame(columns=columns).to_csv(test_path, index=False)

print("✅ Working TRAIN and frozen TEST files initialized")

✅ Working TRAIN and frozen TEST files initialized


In [ ]:
from collections import defaultdict

current_train = defaultdict(int)
current_test = defaultdict(int)

In [12]:
for file in tqdm(csv_files, desc="Streaming & sampling"):
    for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):

        train_rows = []
        test_rows = []

        for _, row in chunk.iterrows():
            label = row[LABEL_COL]

            # Stop early if we've hit the global training size cap
            if sum(current_train.values()) >= MAX_TOTAL_TRAIN_ROWS:
                break

            if current_train[label] >= train_quota[label] and current_test[label] >= test_quota[label]:
                continue

            # decide test vs train
            if current_test[label] < test_quota[label]:
                test_rows.append(row)
                current_test[label] += 1
            elif current_train[label] < train_quota[label]:
                train_rows.append(row)
                current_train[label] += 1

        if train_rows:
            pd.DataFrame(train_rows).to_csv(
                train_path, mode="a", header=False, index=False
            )

        if test_rows:
            pd.DataFrame(test_rows).to_csv(
                test_path, mode="a", header=False, index=False
            )

Streaming & sampling: 100%|██████████| 169/169 [40:51<00:00, 14.51s/it]


In [13]:
train_counts = pd.read_csv(train_path, usecols=[LABEL_COL])[LABEL_COL].value_counts()
test_counts = pd.read_csv(test_path, usecols=[LABEL_COL])[LABEL_COL].value_counts()

verification = pd.DataFrame({
    "train": train_counts,
    "test": test_counts,
    "total": train_counts.add(test_counts, fill_value=0)
}).fillna(0).astype(int)

verification.sort_values("total")

,train,test,total
label,,,
Uploading_Attack,757,250,1007
Recon-PingSweep,1319,448,1767
Backdoor_Malware,1892,638,2530
XSS,2257,766,3023
SqlInjection,3104,1044,4148
CommandInjection,3234,1078,4312
BrowserHijacking,3522,1167,4689
DictionaryBruteForce,7664,2601,10265
DDoS-SlowLoris,13767,4672,18439


In [ ]:
verification.to_csv(f"{OUTPUT_PATH}/working_dataset_verification.csv")
print("✅ Verification saved")

✅ Verification saved


In [ ]:
print("📂 FINAL DATASETS")
print(train_path)
print(test_path)
print(f"{OUTPUT_PATH}/working_dataset_verification.csv")

print("\nApprox TRAIN size:", sum(train_counts))
print("Approx TEST size:", sum(test_counts))

📂 FINAL DATASETS
/content/drive/MyDrive/CIC_IoT_2023/Full_Dataset/train_working.csv
/content/drive/MyDrive/CIC_IoT_2023/Full_Dataset/test_frozen.csv
/content/drive/MyDrive/CIC_IoT_2023/Full_Dataset/working_dataset_verification.csv

Approx TRAIN size: 900000
Approx TEST size: 259645
